If you want to type along with me, head to [this notebook](https://humboldt.cloudbank.2i2c.cloud/hub/user-redirect/git-pull?repo=https%3A%2F%2Fgithub.com%2Fmlekimchi%2Fdata111_fall26&branch=main&urlpath=tree%2Fdata111_fall26%2Flectures%2Flec10_live.ipynb) instead. If you prefer follow along by executing the cells, stay in this notebook.

# Lecture 10: Prediction and Groups
v.ekc

Two big ideas today: making **predictions** from data (using functions + `apply` from last time), and **grouping** — the fastest way to summarize a table. (Slides: KL Lecture 10 deck.)

**Today's sections:**
1. Prediction — child heights
2. Prediction accuracy
3. Grouping by one column
4. Grouping by two columns

In [ ]:
from datascience import *
import numpy as np

%matplotlib inline
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')
import warnings
warnings.filterwarnings("ignore")

---
## 1. Prediction — Child Heights 👨‍👩‍👧

This is Francis Galton's famous 1880s dataset of parents' and children's heights — the data that launched regression. Strategy: to predict a child's height, look at children whose parents were about the same height, and average them.

In [ ]:
families = Table.read_table('family_heights.csv')
families

In [ ]:
parent_avgs = (families.column('father') + families.column('mother'))/2
parent_avgs

In [ ]:
heights = Table().with_columns(
    'Parent Average', parent_avgs,
    'Child', families.column('child'),
    'Sex', families.column('sex')
)
heights

In [ ]:
heights.scatter('Parent Average', 'Child')

In [ ]:
heights.scatter('Parent Average', 'Child')
plots.plot([67.5, 67.5], [50, 85], color='red', lw=2)
plots.plot([68.5, 68.5], [50, 85], color='red', lw=2);

In [ ]:
nearby = heights.where('Parent Average', are.between(67.5, 68.5))
nearby_mean = np.average(nearby.column('Child'))
nearby_mean

In [ ]:
heights.scatter('Parent Average', 'Child')
plots.plot([67.5, 67.5], [50, 85], color='red', lw=2)
plots.plot([68.5, 68.5], [50, 85], color='red', lw=2)
plots.scatter(68, nearby_mean, color='red', s=50);

### Try It 1: A prediction function 😊

1. What if you wanted to predict the child's height for parents with average height 68, 70, and 73? Define a function `predict(h)` that averages the children of parents within ±0.5 inches of `h`.

In [ ]:
def predict(h):
    ...
    return ...

<details>
<summary>💡 Possible answers — click to peek</summary>
<br>

*Filter to the nearby parents, then average their children — exactly what we did by hand for 68.*

```python
def predict(h):
    nearby = heights.where('Parent Average', are.between(h - 0.5, h + 0.5))
    return np.average(nearby.column('Child'))
```

</details>

Run the next cell so the rest of the lecture works:

In [ ]:
# run me: needed below
def predict(h):
    nearby = heights.where('Parent Average', are.between(h - 0.5, h + 0.5))
    return np.average(nearby.column('Child'))

In [ ]:
predict(68)

In [ ]:
predict(70)

In [ ]:
predict(73)

### Try It 2: Predict every row 😊

1. How would you make a prediction for **each row** of parents in the `heights` table? (Hint: yesterday's superpower.)

In [ ]:
# 1. one prediction per row


<details>
<summary>💡 Possible answers — click to peek</summary>
<br>

*`apply` runs `predict` on every Parent Average — 900+ predictions in one line.*

```python
predicted_heights = heights.apply(predict, 'Parent Average')
```

</details>

Run the next cell to continue:

In [ ]:
# run me: needed below
predicted_heights = heights.apply(predict, 'Parent Average')

In [ ]:
heights = heights.with_column('Prediction', predicted_heights)
heights

In [ ]:
heights.select('Parent Average', 'Child', 'Prediction').scatter('Parent Average')

---
## 2. Prediction Accuracy

Good predictions have small errors. Let's *measure* ours — with a function, of course.

In [ ]:
def difference(predicted, actual):
    return predicted - actual

In [ ]:
pred_errs = heights.apply(difference, 'Prediction', 'Child')
heights = heights.with_column('errors',pred_errs)
heights

In [ ]:
heights.hist('errors')

In [ ]:
heights.hist('errors', group='Sex')

### Try It 3: Predict smarter 🤓

The error histogram grouped by sex shows we're systematically off. Can we do better?

1. Write `predict_smarter(h, s)` that only averages nearby children of sex `s`. Does it shrink the errors?

In [ ]:
def predict_smarter(h, s):
    nearby = ...
    nearby_same_sex = ...
    return ...

<details>
<summary>💡 Possible answers — click to peek</summary>
<br>

*Same as `predict`, plus one more `where` for the sex.*

```python
def predict_smarter(h, s):
    nearby = heights.where('Parent Average', are.between(h - 0.5, h + 0.5))
    nearby_same_sex = nearby.where('Sex', s)
    return np.average(nearby_same_sex.column('Child'))
```

</details>

Run the next cell to continue:

In [ ]:
# run me: needed below
def predict_smarter(h, s):
    nearby = heights.where('Parent Average', are.between(h - 0.5, h + 0.5))
    nearby_same_sex = nearby.where('Sex', s)
    return np.average(nearby_same_sex.column('Child'))

In [ ]:
predict_smarter(68, 'female')

In [ ]:
predict_smarter(68, 'male')

In [ ]:
smarter_predicted_heights = heights.apply(predict_smarter, 'Parent Average', 'Sex')
heights = heights.with_column('Smarter Prediction', smarter_predicted_heights)

In [ ]:
smarter_pred_errs = heights.apply(difference, 'Child', 'Smarter Prediction')
heights = heights.with_column('Smarter Errors', smarter_pred_errs)

In [ ]:
heights.hist('Smarter Errors',group='Sex')

> **Compare the two error histograms:** grouping by sex tightened both distributions around 0. Using more relevant data → smarter predictions. That's data science in one sentence.

---
## 3. Grouping by One Column

`group` counted categories in Lecture 8. Give it a **second argument** — a function — and it summarizes each group any way you like.

### 📋 Board Reference

| Code | What it does |
|---|---|
| `t.group('col')` | count rows per category |
| `t.group('col', np.average)` | average of every other column, per category |
| `t.group('col', np.min)` | min per category |
| `t.group(['c1', 'c2'])` | count every combination of two columns |
| `t.apply(f, 'col')` | (from last time) function across a column |

In [ ]:
cones = Table.read_table('cones.csv').drop('Color')
cones

In [ ]:
cones.group('Flavor')

In [ ]:
cones.group('Flavor', np.average)

In [ ]:
cones.group('Flavor', np.min)

Another example

In [ ]:
heights.select('Sex','Child').group('Sex',np.average)

### Grouping the Welcome Survey

Back to our class data — grouping answers real questions fast.

In [ ]:
survey = Table.read_table('data111_survey_fa24.csv')
survey.show(3)

In [ ]:
survey.hist('Extroversion')

In [ ]:
by_extra = survey.group('Extroversion', np.average)
by_extra

In [ ]:
by_extra.plot('Extroversion', 'Texts average')

In [ ]:
survey.group("Sleep Position")

In [ ]:
survey.select("Sleep Position", "Hours of Sleep").group('Sleep Position', np.average)

In [ ]:
survey.select("Sleep Position", "Hours of Sleep").group('Sleep Position', np.average).barh('Sleep Position')

---
### A quick detour: Lists

Arrays hold one type; **lists** (square brackets) hold anything — including the two column names we're about to hand to `group`.

In [ ]:
[1, 5, 'hello', 5.0]

In [ ]:
[1, 5, 'hello', 5.0, make_array(1,2,3)]

---
## 4. Grouping by Two Columns

Pass `group` a *list* of two labels and it counts every combination.

Do right-handed people tend to sleep on their left side and left-handed people sleep on their right?

In [ ]:
survey.group(['Handedness', 'Sleep Position']).show()

### Try It 4: Pets, predicted 🐶

Below is a small `pets` table (8 rows).

1. **Before running anything:** how many rows will `pets.group('Type')` have? What about `pets.group(['Type', 'Color'])`?
2. Check your answers with code.

In [ ]:
pets = Table().with_columns('Type',make_array('dog','cat','dog','dog','bird','cat','dog','bird'),
                            'Color',make_array('black','black','multi','black','multi','black','white','multi'),
                           'Num Legs', make_array(4,4,4,3,2,4,4,2),
                           'Length', make_array(20,16,50,34,8,18,44,7),
                           'Cute score',make_array(8,7,9,7,8,10,3,8)
                           )
pets

In [ ]:
# 2. check your prediction


<details>
<summary>💡 Possible answers — click to peek</summary>
<br>

*One row per distinct value (3 types), one row per distinct **combination** (7 of the 3×3 possible combos actually occur).*

```python
# 2.
pets.group('Type')             # 3 rows
pets.group(['Type', 'Color'])  # 7 rows
```

</details>

---
> **Today's takeaway:** `apply` turns a function into a column; `group` turns categories into summaries. Homework 4 combines both — if the burrito questions feel hard, re-run sections 1 and 3 here first.

## Appendix — Quick Reference

Full `datascience` docs: [data8.org/datascience](https://data8.org/datascience/)

### Minimal template — grouped summary
```python
t.select('category', 'value').group('category', np.average).barh('category')
```